In [ ]:
import numpy as np
import pandas as pd
from collections.abc import Callable
from pathlib import Path
from typing import Any

from numpy.typing import NDArray

import ssms
import lanfactory


In [ ]:
# Specify model
model: str = "angle"

model_dir: Path = Path("../data/torch_models") / model
if not model_dir.exists():
    model_dir = Path("data/torch_models") / model
state_dict_path: Path = next(model_dir.glob("*state_dict*"))
network_config_path: Path = next(model_dir.glob("*network_config*"))

# get_torch_mlp needs the input dimension: params + rt + choice
params: list[str] = ssms.config.model_config[model]["params"]
input_dim: int = len(params) + 2

lan_angle: Callable[[NDArray[np.float32]], Any] = (
    lanfactory.network_inspectors.get_torch_mlp(
        model_file_path=str(state_dict_path),
        network_config=str(network_config_path),
        input_dim=input_dim,
    )
)


In [ ]:
# pick a parameter vector for the model and repeat it across 200 rows
parameter_vector: NDArray[np.float32] = np.array(
    ssms.config.model_config[model]["default_params"], dtype=np.float32
)
parameter_matrix: NDArray[np.float32] = np.tile(parameter_vector, (200, 1))


In [ ]:
# Initialize network input
network_input: NDArray[np.float32] = np.zeros(
    (parameter_matrix.shape[0], parameter_matrix.shape[1] + 2), dtype=np.float32
)
network_input[:, :-2] = parameter_matrix

# Add reaction times
network_input[:, -2] = np.linspace(0, 3, parameter_matrix.shape[0])

# Add choices
network_input[:, -1] = np.repeat(np.random.choice([-1, 1]), parameter_matrix.shape[0])

# Show example output
print("Some network outputs")
print(lan_angle(network_input)[:10])
print("Shape")
print(lan_angle(network_input).shape)


In [ ]:
# make 10 reproducible random parameter sets within the model's bounds
rng: np.random.Generator = np.random.default_rng(123)
lb, ub = ssms.config.model_config[model]["param_bounds"]
parameter_df: pd.DataFrame = pd.DataFrame(
    rng.uniform(lb, ub, size=(10, len(params))), columns=params
)

lanfactory.network_inspectors.kde_vs_lan_likelihoods(
    parameter_df=parameter_df,
    model=model,
    torch_mlp_predict=lan_angle,
    n_samples=2000,
    n_reps=10,
    plot=lanfactory.network_inspectors.PlotConfig(cols=3, show=True),
)


In [ ]:
# use a deterministic parameter vector for the manifold plot
manifold_parameter_df: pd.DataFrame = pd.DataFrame(
    [ssms.config.model_config[model]["default_params"]], columns=params
)

lanfactory.network_inspectors.lan_manifold(
    parameter_df=manifold_parameter_df,
    vary_dict={"v": np.linspace(-2, 2, 20)},
    model=model,
    torch_mlp_predict=lan_angle,
    grid=lanfactory.network_inspectors.GridSpec(n_rt_steps=300, max_rt=5),
    plot=lanfactory.network_inspectors.PlotConfig(
        fig_scale=1.0, save=True, show=True
    ),
)
